# Phonons — lattice vibrations from any interatomic potential

### Markus J. Buehler, MIT

Atoms in a crystal vibrate in collective waves — **phonons**. This module builds
the **dynamical matrix** straight from a pair potential, diagonalizes it, and
plots the **phonon dispersion** $\omega(k)$. A 1D chain checks the method against
the textbook result; a 2D triangular lattice gives the acoustic branches along
$\Gamma\!-\!M\!-\!K\!-\!\Gamma$. The final cell **animates the eigenmodes** live in the
browser.

The point of interest for ML: we compare **Morse** with a trained
**neural-network MLIP** (a bundled fit; paste your own) and ask *does the learned
potential reproduce the vibrations?* — a stringent test, since phonons depend on
the **second** derivative (curvature) of the energy, not just the energy itself.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lamm-mit/MolecularDynamicsModules/blob/main/Phonons.ipynb)


## 1. Potentials and their curvature

Phonon frequencies come from the **curvature** $V''(r)$ of the bonds. We define the
energy $V(r)$ (with the smooth cutoff at $r_c=2.5$) for Morse and the tanh-MLP
**MLIP**, and get $V'$, $V''$ by central finite differences — so the exact same code
handles any potential, learned or analytic.

In [ ]:
import json, warnings, html as _html_mod
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
print('numpy', np.__version__)

RCUT = 2.5
def _cut(r):
    return 0.5 * (np.cos(np.pi * r / RCUT) + 1.0) if r < RCUT else 0.0

def V_morse(r, De=1.0, a=5.0, re=1.0):
    if r >= RCUT: return 0.0
    e = np.exp(-a * (r - re))
    return (De * (1 - e) ** 2 - De) * _cut(r)

DEFAULT_MLIP_WEIGHTS = json.loads(r'''{"mode": "neural_pair_mlp", "w": [20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0], "b_in": [-11.0, -11.506329113924052, -12.012658227848103, -12.518987341772153, -13.025316455696203, -13.531645569620254, -14.037974683544302, -14.544303797468354, -15.050632911392405, -15.556962025316457, -16.06329113924051, -16.569620253164558, -17.075949367088608, -17.582278481012658, -18.088607594936708, -18.594936708860757, -19.10126582278481, -19.60759493670886, -20.11392405063291, -20.62025316455696, -21.12658227848101, -21.632911392405063, -22.139240506329113, -22.645569620253163, -23.151898734177216, -23.65822784810127, -24.164556962025316, -24.670886075949365, -25.17721518987342, -25.68354430379747, -26.189873417721515, -26.696202531645568, -27.202531645569618, -27.708860759493668, -28.21518987341772, -28.72151898734177, -29.22784810126582, -29.734177215189874, -30.24050632911392, -30.74683544303797, -31.253164556962023, -31.759493670886073, -32.265822784810126, -32.77215189873417, -33.278481012658226, -33.78481012658228, -34.291139240506325, -34.79746835443038, -35.303797468354425, -35.81012658227848, -36.31645569620253, -36.82278481012658, -37.32911392405063, -37.835443037974684, -38.34177215189873, -38.848101265822784, -39.35443037974683, -39.860759493670884, -40.36708860759493, -40.87341772151898, -41.379746835443036, -41.88607594936709, -42.392405063291136, -42.89873417721519, -43.40506329113923, -43.91139240506328, -44.417721518987335, -44.92405063291139, -45.43037974683544, -45.93670886075949, -46.443037974683534, -46.94936708860759, -47.45569620253164, -47.962025316455694, -48.46835443037975, -48.974683544303794, -49.48101265822784, -49.98734177215189, -50.49367088607595, -51.0], "c": [-5.5518717211962, -13.471918609473217, -7.470256365269962, -0.021130138412319786, -6.095181530405412, -0.15543232238586932, -3.126278660063525, -0.3351565019865983, -1.4625438949078877, -0.46086708562600176, -0.44561212223911195, -0.5726942521039191, 0.11296997110347977, -0.550545676712719, 0.27756197195314936, -0.3555739894407318, 0.1624461791542589, -0.054554219303062826, -0.06629261109801285, 0.19707569375809783, -0.18262227228543607, 0.2063154623071332, -0.04555067431258565, -0.007682960204658833, 0.16745835820220703, -0.12995265818827634, 0.13648929739896337, 0.025133976822486206, -0.07029897027809205, 0.1544581984353954, -0.06494594947321045, 0.008398888777497917, 0.1119671550354671, -0.08684751680263889, 0.03872852396974475, 0.0827983226226893, -0.08471164058767522, 0.0397580158638091, 0.06975011375194884, -0.07851847275210729, 0.0312226717304655, 0.06576914644951401, -0.07232162203434236, 0.01764916504719794, 0.0687899249850297, -0.06502329391656651, -0.001027228723597776, 0.07497729054895576, -0.054628710090280634, -0.022962731302445406, 0.08533477618130306, -0.041139808639098226, -0.05944262381225551, 0.101717713282744, 0.0014202901251893026, -0.12407280319091507, 0.08078473831879351, 0.11315324687985126, -0.15945500037804852, -0.06782012493746398, 0.23071904502028356, 0.005940865051578155, -0.2884439261145289, 0.06722481058867938, 0.3473847114393246, -0.14340148747650364, -0.43426271319923876, 0.2162728819955369, 0.6184559777314499, -0.219732059489199, -1.0141209560299025, -0.15536243374092015, 1.5523000439962498, 1.8136398781029142, -0.23090876888385395, -3.32065933608983, -5.869383853819228, -7.342769962996257, -8.019511301898179, -8.292210811009348], "b_out": 8.455443492572279}''')

def V_mlip(r, weights=None):
    if r >= RCUT: return 0.0
    w = weights or DEFAULT_MLIP_WEIGHTS
    a = np.asarray(w['w']); b = np.asarray(w['b_in']); c = np.asarray(w['c']); bo = float(w['b_out'])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return float(np.tanh(a * r + b) @ c + bo) * _cut(r)

POTV = {'Morse': V_morse, 'MLIP': V_mlip}

def derivs(Vf, r, h=1e-4):
    # energy, first and second derivative by central finite differences
    Vm, V0, Vp = Vf(r - h), Vf(r), Vf(r + h)
    return V0, (Vp - Vm) / (2 * h), (Vp - 2 * V0 + Vm) / (h * h)


## 2. 1D monatomic chain — method check

For a 1D chain (spacing $a_0$, mass $m=1$) the dispersion is
$\omega(k)^2 = \tfrac1m\sum_{n\ge1} 2\,V''(n a_0)\,[1-\cos(n k a_0)]$, which for
nearest-neighbours only reduces to the classic
$\omega(k) = 2\sqrt{K/m}\,\lvert\sin(k a_0/2)\rvert$ with $K=V''(a_0)$. The acoustic
branch must go to zero at $k\to0$.

In [ ]:
def eq_a0_1d(Vf):
    best = (1.0, np.inf)
    for a0 in np.arange(0.85, 1.15, 0.002):
        e = sum(Vf(n * a0) for n in range(1, int(RCUT / a0) + 2))
        if e < best[1]: best = (a0, e)
    return best[0]

def dispersion_1d(Vf, kk, m=1.0):
    a0 = eq_a0_1d(Vf)
    shells = [n for n in range(1, int(RCUT / a0) + 2) if n * a0 < RCUT]
    Kn = {n: derivs(Vf, n * a0)[2] for n in shells}
    w = [np.sqrt(max(sum(2 * Kn[n] * (1 - np.cos(n * k * a0)) / m for n in shells), 0.0)) for k in kk]
    return a0, np.array(w), Kn

a0 = eq_a0_1d(V_morse); kk = np.linspace(0, np.pi / a0, 200)
a0, wM, Kn = dispersion_1d(V_morse, kk)
_, wL, _   = dispersion_1d(V_mlip, kk)
analytic = 2 * np.sqrt(Kn[1]) * np.abs(np.sin(kk * a0 / 2))
print(f"a0 = {a0:.3f},  K = V''(a0) = {Kn[1]:.2f},  neighbour shells used: {list(Kn)}")

plt.figure(figsize=(6.2, 3.6))
plt.plot(kk * a0 / np.pi, analytic, 'k--', lw=1, label='analytic (NN only)')
plt.plot(kk * a0 / np.pi, wM, 'C0-', lw=2, label='Morse (all shells)')
plt.plot(kk * a0 / np.pi, wL, 'C3-', lw=2, label='MLIP (all shells)')
plt.xlabel(r'$k\,a_0/\pi$'); plt.ylabel(r'$\omega$'); plt.title('1D chain phonon dispersion')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 3. 2D triangular lattice — dispersion along $\Gamma\!-\!M\!-\!K\!-\!\Gamma$

In 2D each bond contributes a $2\times2$ force-constant block
$\Phi = (V''-V'/r)\,\hat n\hat n^{\top} + (V'/r)\,\mathbf I$, and the dynamical matrix is
$\mathbf D(\mathbf k)=\tfrac1m\sum_{\text{bonds}}\Phi\,[1-\cos(\mathbf k\!\cdot\!\mathbf R)]$.
Its two eigenvalues give the longitudinal and transverse **acoustic** branches.
Both must vanish at $\Gamma$ (the acoustic sum rule). We overlay **Morse vs MLIP**.

In [ ]:
def eq_a0_2d(Vf):
    best = (1.0, np.inf)
    for a0 in np.arange(0.90, 1.06, 0.002):
        e = 0.0
        for i in range(-3, 4):
            for j in range(-3, 4):
                if i == 0 and j == 0: continue
                Rx = i * a0 + j * a0 * 0.5; Ry = j * a0 * np.sqrt(3) / 2; r = np.hypot(Rx, Ry)
                if r < RCUT: e += 0.5 * Vf(r)
        if e < best[1]: best = (a0, e)
    return best[0]

def neighbors_2d(a0):
    out = []
    for i in range(-3, 4):
        for j in range(-3, 4):
            if i == 0 and j == 0: continue
            Rx = i * a0 + j * a0 * 0.5; Ry = j * a0 * np.sqrt(3) / 2; r = np.hypot(Rx, Ry)
            if 1e-6 < r < RCUT: out.append((Rx, Ry, r))
    return out

def Dk(Vf, a0, kx, ky, m=1.0):
    D = np.zeros((2, 2))
    for Rx, Ry, r in neighbors_2d(a0):
        _, Vp, Vpp = derivs(Vf, r)
        n = np.array([Rx, Ry]) / r
        Phi = (Vpp - Vp / r) * np.outer(n, n) + (Vp / r) * np.eye(2)
        D += Phi * (1 - np.cos(kx * Rx + ky * Ry))
    return D / m

def kpath(a0, npts=60):
    b1 = 2 * np.pi / a0 * np.array([1.0, -1 / np.sqrt(3)])
    b2 = 2 * np.pi / a0 * np.array([0.0, 2 / np.sqrt(3)])
    G = np.array([0, 0.]); M = 0.5 * b1; K = (b1 + b2) / 3.0
    pts = [G, M, K, G]; ks = []; ticks = [0]
    for s in range(3):
        for t in np.linspace(0, 1, npts, endpoint=False): ks.append(pts[s] + t * (pts[s + 1] - pts[s]))
        ticks.append(len(ks))
    ks.append(G)
    return np.array(ks), ticks

def dispersion_2d(Vf):
    a0 = eq_a0_2d(Vf); ks, ticks = kpath(a0)
    om = np.array([np.sqrt(np.clip(np.linalg.eigvalsh(Dk(Vf, a0, k[0], k[1]), ), 0, None)) for k in ks])
    return a0, om, ticks

a0M, omM, ticks = dispersion_2d(V_morse)
a0L, omL, _     = dispersion_2d(V_mlip)
print(f"Morse a0={a0M:.3f}, MLIP a0={a0L:.3f};  omega(Gamma)={omM[0].round(3)} (acoustic sum rule)")
print(f"dispersion RMSE (Morse vs MLIP): {np.sqrt(np.mean((omM-omL)**2)):.3f}")

x = np.arange(len(omM))
plt.figure(figsize=(7, 4))
plt.plot(x, omM[:, 0], 'C0-', lw=2, label='Morse'); plt.plot(x, omM[:, 1], 'C0-', lw=2)
plt.plot(x, omL[:, 0], 'C3--', lw=2, label='MLIP'); plt.plot(x, omL[:, 1], 'C3--', lw=2)
for t in ticks: plt.axvline(t, color='gray', lw=0.5)
plt.xticks(ticks, [r'$\Gamma$', 'M', 'K', r'$\Gamma$'])
plt.ylabel(r'$\omega$'); plt.title('2D triangular lattice: phonon dispersion (acoustic branches)')
plt.legend(); plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()


## 4. Interactive eigenmode animation

Pick a wavevector along $\Gamma\!-\!M\!-\!K\!-\!\Gamma$ and a branch, and watch the atoms
oscillate in that phonon. The displacement of each atom is
$\mathbf u(\mathbf R,t)=A\,\hat{\boldsymbol\epsilon}\,\cos(\mathbf k\!\cdot\!\mathbf R-\omega t)$, with
$(\omega,\hat{\boldsymbol\epsilon})$ from the $2\times2$ dynamical matrix computed live in the
browser. The mini-plot shows the dispersion with your current $k$ marked. Switch
the potential to compare **Morse** and the **MLIP**.

In [ ]:
SIMULATOR_HTML = '<!doctype html><html><head><meta charset="utf-8"/><style>\n body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;margin:0;padding:8px;background:#fafafa;font-size:12px;color:#222}\n .row{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:6px}\n label{display:inline-flex;align-items:center;gap:4px;font-size:11px} label.s{min-width:150px} input[type=range]{width:120px}\n button{padding:4px 12px;font-size:12px;cursor:pointer} button.p{background:#2e7d32;color:#fff;border:0;border-radius:3px}\n canvas{background:#fff;border:1px solid #ccc} #st{font-family:monospace;font-size:11px;color:#444}\n</style></head><body>\n<div class="row">\n <button id="go" class="p">Pause</button>\n <label>potential <select id="pot"><option>Morse</option><option>MLIP</option></select></label>\n <label>branch <select id="br"><option value="0">acoustic 1 (lower)</option><option value="1">acoustic 2 (upper)</option></select></label>\n <span id="st"></span>\n</div>\n<div class="row">\n <label class="s">k along &#915;-M-K-&#915; <input type="range" id="kt" min="0" max="1" step="0.005" value="0.33"/><span id="kt-v">0.33</span></label>\n <label class="s">amplitude <input type="range" id="amp" min="0.02" max="0.30" step="0.01" value="0.12"/><span id="amp-v">0.12</span></label>\n <label class="s">speed <input type="range" id="sp" min="0.1" max="3" step="0.1" value="1.0"/><span id="sp-v">1.0</span></label>\n</div>\n<div class="row" style="align-items:flex-start">\n <canvas id="world" width="560" height="430"></canvas>\n <canvas id="disp" width="380" height="430"></canvas>\n</div>\n<script>\nconst RCUT=2.5, MLIP=/*MLIP*/null, $=id=>document.getElementById(id);\nfunction cut(r){return r<RCUT?0.5*(Math.cos(Math.PI*r/RCUT)+1):0;}\nfunction Vmorse(r){if(r>=RCUT)return 0;const e=Math.exp(-5*(r-1));return ((1-e)*(1-e)-1)*cut(r);}\nfunction Vmlip(r){if(r>=RCUT||!MLIP)return 0;const w=MLIP.w,b=MLIP.b_in,c=MLIP.c;let s=MLIP.b_out;for(let j=0;j<c.length;j++)s+=c[j]*Math.tanh(w[j]*r+b[j]);return s*cut(r);}\nfunction Vof(r){return $(\'pot\').value===\'Morse\'?Vmorse(r):Vmlip(r);}\nfunction der(r){const h=1e-4,Vm=Vof(r-h),V0=Vof(r),Vp=Vof(r+h);return [(Vp-Vm)/(2*h),(Vp-2*V0+Vm)/(h*h)];}\nlet A0=1, NB=[];\nfunction eqA0(){let bA=1,bE=1e9;for(let a=0.9;a<=1.06;a+=0.004){let e=0;\n  for(let i=-3;i<=3;i++)for(let j=-3;j<=3;j++){if(!i&&!j)continue;const Rx=i*a+j*a*0.5,Ry=j*a*Math.sqrt(3)/2,r=Math.hypot(Rx,Ry);if(r<RCUT)e+=0.5*Vof(r);}\n  if(e<bE){bE=e;bA=a;}}return bA;}\nfunction nbrs(a){const o=[];for(let i=-3;i<=3;i++)for(let j=-3;j<=3;j++){if(!i&&!j)continue;const Rx=i*a+j*a*0.5,Ry=j*a*Math.sqrt(3)/2,r=Math.hypot(Rx,Ry);if(r>1e-6&&r<RCUT)o.push([Rx,Ry,r]);}return o;}\nfunction Dk(kx,ky){let a=0,b=0,d=0;for(const nb of NB){const Rx=nb[0],Ry=nb[1],r=nb[2];const dd=der(r),Vp=dd[0],Vpp=dd[1];\n  const nx=Rx/r,ny=Ry/r,w=(1-Math.cos(kx*Rx+ky*Ry));const t=Vpp-Vp/r,iso=Vp/r;\n  a+=(t*nx*nx+iso)*w;b+=(t*nx*ny)*w;d+=(t*ny*ny+iso)*w;}return [a,b,d];}\nfunction eig2(D){const a=D[0],b=D[1],d=D[2];const tr=(a+d)/2,de=Math.sqrt(((a-d)/2)**2+b*b);\n  const l0=tr-de,l1=tr+de;let v0,v1;\n  function vec(l){if(Math.abs(b)>1e-9){const v=[l-d,b],n=Math.hypot(v[0],v[1]);return [v[0]/n,v[1]/n];}return (l<=a)?[1,0]:[0,1];}\n  return [[Math.sqrt(Math.max(l0,0)),vec(l0)],[Math.sqrt(Math.max(l1,0)),vec(l1)]];}\nfunction kpath(a){const b1=[2*Math.PI/a,2*Math.PI/a*(-1/Math.sqrt(3))],b2=[0,2*Math.PI/a*(2/Math.sqrt(3))];\n  const G=[0,0],M=[0.5*b1[0],0.5*b1[1]],K=[(b1[0]+b2[0])/3,(b1[1]+b2[1])/3];return [G,M,K,G];}\nfunction kAt(t){const P=kpath(A0);const seg=Math.min(2,Math.floor(t*3)),f=t*3-seg;return [P[seg][0]+f*(P[seg+1][0]-P[seg][0]),P[seg][1]+f*(P[seg+1][1]-P[seg][1])];}\n// lattice for rendering\nlet R0=[],NX=14,NY=12,Lx=1,Ly=1;\nfunction buildLattice(){A0=eqA0();NB=nbrs(A0);R0=[];const dy=A0*Math.sqrt(3)/2;\n  for(let j=0;j<NY;j++)for(let i=0;i<NX;i++)R0.push([i*A0+(j%2?A0/2:0),j*dy]);Lx=NX*A0;Ly=(NY-1)*dy;drawDisp();}\nconst cv=$(\'world\'),cx=cv.getContext(\'2d\'),dc=$(\'disp\'),dx=dc.getContext(\'2d\');\nlet t0=performance.now(),run=true;\nfunction frame(){const now=performance.now(),tt=(now-t0)/1000*(+$(\'sp\').value);\n  const k=kAt(+$(\'kt\').value),ev=eig2(Dk(k[0],k[1])),br=+$(\'br\').value,om=ev[br][0],pol=ev[br][1],amp=+$(\'amp\').value;\n  cx.clearRect(0,0,cv.width,cv.height);cx.fillStyle=\'#fff\';cx.fillRect(0,0,cv.width,cv.height);\n  const pad=24,sc=Math.min((cv.width-2*pad)/Lx,(cv.height-2*pad)/Ly);\n  for(let n=0;n<R0.length;n++){const Rx=R0[n][0],Ry=R0[n][1];const ph=k[0]*Rx+k[1]*Ry-om*tt;\n    const ux=amp*pol[0]*Math.cos(ph),uy=amp*pol[1]*Math.cos(ph);\n    const X=pad+(Rx+ux)*sc,Y=cv.height-pad-(Ry+uy)*sc;const q=Math.min(1,Math.hypot(ux,uy)/amp);\n    cx.fillStyle=\'rgb(\'+Math.round(40+215*q)+\',90,\'+Math.round(200-150*q)+\')\';\n    cx.beginPath();cx.arc(X,Y,Math.max(2.5,sc*0.32),0,6.2832);cx.fill();}\n  cx.fillStyle=\'#222\';cx.font=\'12px monospace\';\n  cx.fillText($(\'pot\').value+\'  k=[\'+k[0].toFixed(2)+\',\'+k[1].toFixed(2)+\']  omega=\'+om.toFixed(2),8,16);\n  if(run)requestAnimationFrame(frame);}\nfunction drawDisp(){dx.clearRect(0,0,dc.width,dc.height);dx.fillStyle=\'#fff\';dx.fillRect(0,0,dc.width,dc.height);\n  const npts=120,W=dc.width-50,H=dc.height-40,x0=40,y0=12;let om=[];let mx=1e-9;\n  for(let s=0;s<npts;s++){const t=s/(npts-1);const k=kAt(t),ev=eig2(Dk(k[0],k[1]));om.push([ev[0][0],ev[1][0]]);mx=Math.max(mx,ev[1][0]);}\n  dx.strokeStyle=\'#ddd\';dx.strokeRect(x0,y0,W,H);\n  for(let bch=0;bch<2;bch++){dx.strokeStyle=bch?\'#1f77b4\':\'#2e7d32\';dx.beginPath();\n    om.forEach((o,s)=>{const X=x0+W*s/(npts-1),Y=y0+H-(o[bch]/mx)*H;s?dx.lineTo(X,Y):dx.moveTo(X,Y);});dx.stroke();}\n  const tc=+$(\'kt\').value,Xc=x0+W*tc;dx.strokeStyle=\'#c0392b\';dx.beginPath();dx.moveTo(Xc,y0);dx.lineTo(Xc,y0+H);dx.stroke();\n  dx.fillStyle=\'#333\';dx.font=\'11px sans-serif\';dx.fillText(\'omega\',4,12);\n  [[\'G\',0],[\'M\',1/3],[\'K\',2/3],[\'G\',1]].forEach(p=>dx.fillText(p[0],x0+W*p[1]-3,y0+H+14));}\n$(\'go\').onclick=function(){run=!run;$(\'go\').textContent=run?\'Pause\':\'Play\';if(run){t0=performance.now();frame();}};\n[\'kt\',\'amp\',\'sp\'].forEach(id=>$(id).addEventListener(\'input\',function(){$(id+\'-v\').textContent=(+$(id).value).toFixed(2);drawDisp();}));\n$(\'pot\').addEventListener(\'change\',buildLattice);$(\'br\').addEventListener(\'change\',()=>{});\nbuildLattice();frame();\n</script></body></html>'

def show_phonons():
    mlip_js = json.dumps({'w': DEFAULT_MLIP_WEIGHTS['w'], 'b_in': DEFAULT_MLIP_WEIGHTS['b_in'],
                          'c': DEFAULT_MLIP_WEIGHTS['c'], 'b_out': DEFAULT_MLIP_WEIGHTS['b_out']})
    html = SIMULATOR_HTML.replace('/*MLIP*/null', mlip_js)
    esc = _html_mod.escape(html, quote=True)
    display({'text/html': f'<iframe srcdoc="{esc}" width="980" height="520" '
             'style="border:1px solid #ddd;border-radius:6px;"></iframe>'}, raw=True)

show_phonons()
